# Customer Churn Analysis
EDA, статистический анализ и визуализация.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/customer_churn.csv')
df.head()


## 1. Качество данных

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:')
print(df.isna().sum().sort_values(ascending=False))
print('\nDuplicate rows:', df.duplicated().sum())


In [ ]:
df = df.drop_duplicates().copy()
df['age'] = df['age'].fillna(df['age'].median())
df['monthly_charges'] = df['monthly_charges'].fillna(df['monthly_charges'].median())
df['total_charges'] = df['total_charges'].fillna(df['monthly_charges'] * df['tenure_months'])
df['payment_method'] = df['payment_method'].fillna('Unknown')
df['churn_flag'] = (df['churn'] == 'Yes').astype(int)
print(df.shape)


## 2. Churn rate и EDA

In [ ]:
churn_rate = df['churn_flag'].mean() * 100
print(f'Overall churn rate: {churn_rate:.2f}%')

contract_churn = (df.groupby('contract')['churn_flag'].mean()*100).sort_values(ascending=False)
print(contract_churn)


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.barplot(data=df, x='contract', y='churn_flag', estimator='mean', errorbar=None, ax=ax)
ax.set_ylabel('Churn rate')
ax.set_xlabel('Contract')
ax.set_title('Churn rate by contract type')
ax.yaxis.set_major_formatter(lambda x, pos: f'{x:.0%}')
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=df, x='churn', y='monthly_charges', ax=ax)
ax.set_title('Monthly charges by churn status')
plt.tight_layout()
plt.show()


## 3. Статистические гипотезы

In [ ]:
# Chi-square: contract vs churn
ct = pd.crosstab(df['contract'], df['churn'])
chi2, p_value, dof, expected = stats.chi2_contingency(ct)
print(f'Chi-square={chi2:.2f}, p-value={p_value:.6g}')

# Mann-Whitney U: monthly charges for churned vs retained customers
x = df.loc[df['churn']=='Yes', 'monthly_charges']
y = df.loc[df['churn']=='No', 'monthly_charges']
u_stat, p_u = stats.mannwhitneyu(x, y, alternative='two-sided')
print(f'Mann-Whitney U={u_stat:.0f}, p-value={p_u:.6g}')


## 4. Итоговые выводы
Интерпретировать p-value вместе с размером эффекта и бизнес-контекстом. Не делать причинных выводов из корреляционных/наблюдательных данных.